In [1]:
import os 
import random

import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
from datetime import datetime, timedelta

In [2]:
random.seed(24)

In [3]:
pd.set_option('display.max_columns', None)

# Stroke data

In [4]:
stroke_data_final = pd.DataFrame(columns=['patient_id','concept','value','timestamp','type','visit_label'])

In [5]:
stroke_data_final

,patient_id,concept,value,timestamp,type,visit_label


In [6]:
patients  = pd.read_csv('../../eyescore/data/extracts/stroke/PatientDim.csv')
encounters = pd.read_csv('../../eyescore/data/extracts/stroke/EncounterFact.csv')
diagnoses  = pd.read_csv('../../eyescore/data/extracts/stroke/DiagnosisEventFact.csv')
medications = pd.read_csv('../../eyescore/data/extracts/stroke/MedicationEventFact.csv',low_memory=False)
procedures =  pd.read_csv('../../eyescore/data/extracts/stroke/ProcedureEventFact.csv')

hemato = pd.read_csv('../../eyescore/data/extracts/stroke/labs/HEMATOCRIT.csv')
hemo = pd.read_csv('../../eyescore/data/extracts/stroke/labs/HEMOGLOBIN.csv')
plat = pd.read_csv('../../eyescore/data/extracts/stroke/labs/PLATELETS.csv')
rbc = pd.read_csv('../../eyescore/data/extracts/stroke/labs/RBC.csv')
wbc = pd.read_csv('../../eyescore/data/extracts/stroke/labs/WBC.csv')

sbp = pd.read_csv('../../eyescore/data/extracts/stroke/vitals/FlowsheetSBP.csv')
dbp = pd.read_csv('../../eyescore/data/extracts/stroke/vitals/FlowsheetDBP.csv')
rr = pd.read_csv('../../eyescore/data/extracts/stroke/vitals/FlowsheetRR.csv')
hr = pd.read_csv('../../eyescore/data/extracts/stroke/vitals/FlowsheetHR.csv')

In [7]:
patient_ids = random.sample(patients.PatientDurableKey.unique().tolist(),k=4000)

In [8]:
encounters  = encounters[encounters.PatientDurableKey.isin(patient_ids)].reset_index(drop=True)
# patients  = patients[patients.PatientDurableKey.isin(patient_ids)].reset_index(drop=True)
diagnoses  = diagnoses[diagnoses.PatientDurableKey.isin(patient_ids)].reset_index(drop=True)
medications = medications[medications.PatientDurableKey.isin(patient_ids)].reset_index(drop=True)
procedures =  procedures[procedures.PatientDurableKey.isin(patient_ids)].reset_index(drop=True)


hemato = hemato[hemato.PatientDurableKey.isin(patient_ids)]
hemo = hemo[hemo.PatientDurableKey.isin(patient_ids)]
plat = plat[plat.PatientDurableKey.isin(patient_ids)]
rbc = rbc[rbc.PatientDurableKey.isin(patient_ids)]
wbc = wbc[wbc.PatientDurableKey.isin(patient_ids)]

sbp = sbp[sbp.PatientDurableKey.isin(patient_ids)]
dbp = dbp[dbp.PatientDurableKey.isin(patient_ids)]
rr = rr[rr.PatientDurableKey.isin(patient_ids)]
hr = hr[hr.PatientDurableKey.isin(patient_ids)]


In [9]:
plat = plat[plat.PLATELETS != 'Clumped  Platelets']
plat = plat[plat.PLATELETS != 'CLUMPS']
plat = plat[plat.PLATELETS != 'See comment']
plat = plat[plat.PLATELETS != 'Platelet Clumping']
plat = plat[plat.PLATELETS != 'Platelet clumping']

hemato = hemato[hemato.HEMATOCRIT !='See comment']
hemo = hemo[hemo.HEMOGLOBIN !='See comment']
rbc = rbc[rbc.RBC != 'See comment']
wbc = wbc[wbc.WBC != 'See comment']

In [10]:
encounters = encounters.groupby('PatientDurableKey').first().reset_index()

encounters = encounters[['PatientDurableKey',
                         'EncounterKey',
                         'PatientAge',
                         'EncounterStartInstant',
                         'EncounterEndInstant',
                         'LengthOfStayDays']]

encounter_ids = encounters.EncounterKey.unique()

In [11]:
encounters

,PatientDurableKey,EncounterKey,PatientAge,EncounterStartInstant,EncounterEndInstant,LengthOfStayDays
0,17,3066557,72.0,2017-02-16T16:36:42Z,2017-04-20T19:11:00Z,63.11
1,38,3159889,70.0,2017-11-24T20:02:53Z,2017-11-27T12:49:00Z,2.70
2,71,3395023,38.0,2017-02-18T03:09:33Z,2017-02-18T04:51:00Z,0.07
3,273,13057380,56.0,2021-12-01T06:54:00Z,2021-12-03T11:56:00Z,2.21
4,309,1596690,60.0,2019-05-19T20:45:17Z,2019-05-30T12:23:00Z,10.65
...,...,...,...,...,...,...
3995,1265651,19266570,61.0,2023-07-23T19:32:00Z,2023-08-16T09:40:00Z,23.59
3996,1265681,19267105,49.0,2023-07-23T17:33:00Z,2023-07-27T12:15:00Z,3.78
3997,1268204,19324782,45.0,2023-07-27T15:29:00Z,2023-07-29T12:45:00Z,1.89
3998,1268232,19319049,41.0,2023-07-27T21:25:00Z,2023-07-29T08:44:00Z,1.47


In [12]:
# patients  = patients[patients.EncounterKey.isin(encounter_ids)].reset_index(drop=True)
diagnoses  = diagnoses[diagnoses.EncounterKey.isin(encounter_ids)].reset_index(drop=True)
medications = medications[medications.EncounterKey.isin(encounter_ids)].reset_index(drop=True)
procedures =  procedures[procedures.EncounterKey.isin(encounter_ids)].reset_index(drop=True)


hemato = hemato[hemato.EncounterKey.isin(encounter_ids)]
hemo = hemo[hemo.EncounterKey.isin(encounter_ids)]
plat = plat[plat.EncounterKey.isin(encounter_ids)]
rbc = rbc[rbc.EncounterKey.isin(encounter_ids)]
wbc = wbc[wbc.EncounterKey.isin(encounter_ids)]

sbp = sbp[sbp.EncounterKey.isin(encounter_ids)]
dbp = dbp[dbp.EncounterKey.isin(encounter_ids)]
rr = rr[rr.EncounterKey.isin(encounter_ids)]
hr = hr[hr.EncounterKey.isin(encounter_ids)]

In [13]:
diagnoses.head(10)

diagnoses = diagnoses.groupby(['PatientDurableKey','EncounterKey','DiagnosisKey']).first().reset_index()
diagnoses = diagnoses.groupby(['PatientDurableKey','EncounterKey','DiagnosisCode']).first().reset_index()

diagnoses = diagnoses[['PatientDurableKey',
                       'EncounterKey',
                       'EncounterStartInstant',
                       'DiagnosisEventUserEnteredDateKey',
                       'DiagnosisCode']]

In [14]:
diagnoses = diagnoses.rename(columns={'PatientDurableKey':'patient_id',
                                      'DiagnosisCode':'concept',
                                      'DiagnosisEventUserEnteredDateKey':'timestamp'})


In [15]:
diagnoses['value'] = np.nan
diagnoses['type'] = 'diagnosis'
diagnoses['visit_label'] = np.nan
diagnoses = diagnoses[['patient_id','concept','value','timestamp','type','visit_label']]

In [16]:
medications = medications.groupby(['PatientDurableKey','EncounterKey','MedicationKey']).first().reset_index()
medications = medications.groupby(['PatientDurableKey','EncounterKey','MedicationName']).first().reset_index()


medications = medications[['PatientDurableKey',
                           'EncounterKey',
                           'MedicationName',
                           'MedicationEventStartInstant']]

In [17]:
# medications['MedicationEventStartInstant'][0][0:10]
medications.MedicationEventStartInstant = medications.MedicationEventStartInstant.apply(lambda x: x[0:10])

In [18]:
medications = medications.rename(columns={'PatientDurableKey':'patient_id',
                                      'MedicationName':'concept',
                                      'MedicationEventStartInstant':'timestamp'})

medications['value'] = np.nan
medications['type'] = 'medication'
medications['visit_label'] = np.nan
medications = medications[['patient_id','concept','value','timestamp','type','visit_label']]

In [19]:
# procedures = procedures.groupby(['PatientDurableKey','EncounterKey','ProcedureKey']).first().reset_index()
# procedures = procedures.groupby(['PatientDurableKey','EncounterKey','ProcedureName']).first().reset_index()

# procedures = procedures[['PatientDurableKey',
#                            'EncounterKey',
#                            'ProcedureName',
#                            'ProcedureEventStartInstant']]

# procedures.ProcedureEventStartInstant = procedures.ProcedureEventStartInstant.apply(lambda x: x[0:10])

In [20]:
# procedures = procedures.rename(columns={'PatientDurableKey':'patient_id',
#                                       'ProcedureName':'concept',
#                                       'ProcedureEventStartInstant':'timestamp'})

# procedures['value'] = np.nan
# procedures['type'] = 'procedure'
# procedures['visit_label'] = np.nan
# procedures = procedures[['patient_id','concept','value','timestamp','type','visit_label']]

In [21]:
# procedures

In [22]:
hemato = hemato.rename(columns={'PatientDurableKey':'patient_id',
                        'EncounterStartInstant':'timestamp',
                        'HEMATOCRIT':'value'})

hemato['value'] = pd.to_numeric(hemato['value'])
hemato = hemato.groupby(['patient_id','timestamp']).mean().reset_index()
hemato.timestamp = hemato.timestamp.apply(lambda x: x[0:10])

hemato['concept'] = 'labs_hemato'
hemato['type'] = 'labs'
hemato['visit_label'] = np.nan
hemato = hemato[['patient_id','concept','value','timestamp','type','visit_label']]

In [23]:
hemo = hemo.rename(columns={'PatientDurableKey':'patient_id',
                            'EncounterStartInstant':'timestamp',
                            'HEMOGLOBIN':'value'})

hemo['value'] = pd.to_numeric(hemo['value'])
hemo = hemo.groupby(['patient_id','timestamp']).mean().reset_index()
hemo.timestamp = hemo.timestamp.apply(lambda x: x[0:10])

hemo['concept'] = 'labs_hemo'
hemo['type'] = 'labs'
hemo['visit_label'] = np.nan
hemo = hemo[['patient_id','concept','value','timestamp','type','visit_label']]

In [24]:
plat = plat.rename(columns={'PatientDurableKey':'patient_id',
                            'EncounterStartInstant':'timestamp',
                            'PLATELETS':'value'})

plat['value'] = pd.to_numeric(plat['value'])
plat = plat.groupby(['patient_id','timestamp']).mean().reset_index()
plat.timestamp = plat.timestamp.apply(lambda x: x[0:10])

plat['concept'] = 'labs_plat'
plat['type'] = 'labs'
plat['visit_label'] = np.nan
plat = plat[['patient_id','concept','value','timestamp','type','visit_label']]

In [25]:
rbc = rbc.rename(columns={'PatientDurableKey':'patient_id',
                            'EncounterStartInstant':'timestamp',
                            'RBC':'value'})

rbc['value'] = pd.to_numeric(rbc['value'])
rbc = rbc.groupby(['patient_id','timestamp']).mean().reset_index()
rbc.timestamp = rbc.timestamp.apply(lambda x: x[0:10])
rbc['concept'] = 'labs_rbc'
rbc['type'] = 'labs'
rbc['visit_label'] = np.nan
rbc = rbc[['patient_id','concept','value','timestamp','type','visit_label']]

In [26]:
wbc = wbc.rename(columns={'PatientDurableKey':'patient_id',
                          'EncounterStartInstant':'timestamp',
                          'WBC':'value'})

wbc['value'] = pd.to_numeric(wbc['value'])
wbc = wbc.groupby(['patient_id','timestamp']).mean().reset_index()
wbc.timestamp = wbc.timestamp.apply(lambda x: x[0:10])

wbc['concept'] = 'labs_wbc'
wbc['type'] = 'labs'
wbc['visit_label'] = np.nan
wbc = wbc[['patient_id','concept','value','timestamp','type','visit_label']]

In [27]:
sbp = sbp.rename(columns={'PatientDurableKey':'patient_id',
                          'FlowsheetValueTakenDate':'timestamp',
                          'SBP':'value'})

sbp = sbp.groupby(['patient_id','timestamp']).mean().reset_index()
sbp['concept'] = 'vitals_sbp'
sbp['type'] = 'vitals'
sbp['visit_label'] = np.nan
sbp = sbp[['patient_id','concept','value','timestamp','type','visit_label']]

In [28]:
dbp = dbp.rename(columns={'PatientDurableKey':'patient_id',
                          'FlowsheetValueTakenDate':'timestamp',
                          'DBP':'value'})

dbp = dbp.groupby(['patient_id','timestamp']).mean().reset_index()
dbp['concept'] = 'vitals_dbp'
dbp['type'] = 'vitals'
dbp['visit_label'] = np.nan
dbp = dbp[['patient_id','concept','value','timestamp','type','visit_label']]

In [29]:
rr = rr.rename(columns={'PatientDurableKey':'patient_id',
                          'FlowsheetValueTakenDate':'timestamp',
                          'RR':'value'})

rr = rr.groupby(['patient_id','timestamp']).mean().reset_index()
rr['concept'] = 'vitals_rr'
rr['type'] = 'vitals'
rr['visit_label'] = np.nan
rr = rr[['patient_id','concept','value','timestamp','type','visit_label']]

In [30]:
hr = hr.rename(columns={'PatientDurableKey':'patient_id',
                          'FlowsheetValueTakenDate':'timestamp',
                          'HR':'value'})

hr = hr.groupby(['patient_id','timestamp']).mean().reset_index()
hr['concept'] = 'vitals_hr'
hr['type'] = 'vitals'
hr['visit_label'] = np.nan
hr = hr[['patient_id','concept','value','timestamp','type','visit_label']]

In [31]:
all_patients = []

for i, patient in tqdm(enumerate(encounters.PatientDurableKey)):

    d = diagnoses[diagnoses.patient_id == patient] 
    m = medications[medications.patient_id == patient]
    hc = hemato[hemato.patient_id == patient]
    he = hemo[hemo.patient_id == patient]
    pl = plat[plat.patient_id == patient]
    rb = rbc[rbc.patient_id == patient]
    wb = wbc[wbc.patient_id == patient]
    sb = sbp[sbp.patient_id == patient]
    db = dbp[dbp.patient_id == patient]
    r = rr[rr.patient_id == patient]
    h = hr[hr.patient_id == patient]
    
    x = pd.concat([d,m,hc,he,pl,rb,wb,sb,db,r,h]).sort_values('timestamp').reset_index(drop=True)
    x['LOS'] = float(encounters[encounters.PatientDurableKey == patient].LengthOfStayDays[i])
    x['visit_label'] = x.LOS.apply(lambda x: 1 if x > 30 else 0)
    
    all_patients.append(x)

0it [00:00, ?it/s]

In [32]:
final = pd.concat(all_patients)

In [89]:
final['dummy'] = 1

In [90]:
final.groupby('patient_id').sum().dummy.min()

np.int64(8)

In [91]:
final.to_csv('dataset4k-30-days.csv')

In [38]:
diagnoses.patient_id.unique()

array([     17,      38,      71, ..., 1268204, 1268232, 1269380])

In [39]:
final[final.patient_id == 71]

,patient_id,concept,value,timestamp,type,visit_label,LOS
0,71,ASPIRIN 81 MG CHEWABLE TABLET,NaN,2017-02-17,medication,0,0.07
1,71,IODIXANOL 320 MG IODINE/ML INJECTION SOLUTION,NaN,2017-02-18,medication,0,0.07
2,71,IODIXANOL 320 MG IODINE/ML INTRAVENOUS SOLUTION,NaN,2017-02-18,medication,0,0.07
3,71,labs_hemato,0.460000,2017-02-18,labs,0,0.07
4,71,labs_hemo,155.000000,2017-02-18,labs,0,0.07
5,71,labs_plat,225.000000,2017-02-18,labs,0,0.07
6,71,labs_rbc,5.120000,2017-02-18,labs,0,0.07
7,71,labs_wbc,7.800000,2017-02-18,labs,0,0.07
8,71,vitals_sbp,128.000000,2017-02-18,vitals,0,0.07
9,71,vitals_dbp,79.666667,2017-02-18,vitals,0,0.07


In [36]:
final.patient_id.unique()

array([     17,      38,      71, ..., 1268204, 1268232, 1269380])